In [ ]:
# ----- Verify credentials (optional) -----
import boto3

def get_iam_identity():
    session = boto3.session.Session()
    credentials = session.get_credentials()
    if credentials is None:
        print("No credentials were found.")
    else:
        print("Found credentials, method:", credentials.method)
    sts_client = session.client('sts')
    try:
        identity = sts_client.get_caller_identity()
        print("Caller Identity:", identity)
    except Exception as e:
        print("Error retrieving caller identity:", e)

get_iam_identity()


Found credentials, method: env
Caller Identity: {'UserId': 'AIDA6G75DLDQBJJJ7TAM7', 'Account': '977098987744', 'Arn': 'arn:aws:iam::977098987744:user/student_projects', 'ResponseMetadata': {'RequestId': '4b6f90d7-dcc3-4509-8c4a-8bca57686d62', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amzn-requestid': '4b6f90d7-dcc3-4509-8c4a-8bca57686d62', 'content-type': 'text/xml', 'content-length': '413', 'date': 'Mon, 14 Apr 2025 05:45:21 GMT'}, 'RetryAttempts': 0}}


In [ ]:
import os
import boto3
import io
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
import optuna

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import timm
import optuna
import pandas as pd
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR

In [ ]:

# ----- Custom Dataset for Direct S3 Access -----
class S3ImageFolder(Dataset):
    def __init__(self, bucket_name, prefix, transform=None, s3_client=None):

        self.bucket_name = bucket_name
        self.prefix = prefix
        self.transform = transform
        self.s3_client = s3_client or boto3.client('s3')
        self.classes, self.class_to_idx = self._find_classes()
        self.samples = self._make_dataset()

    def _find_classes(self):
        """
        Discover class subdirectories under the given prefix.
        Returns:
            classes (list): Sorted list of class names.
            class_to_idx (dict): Mapping from class names to indices.
        """
        paginator = self.s3_client.get_paginator('list_objects_v2')
        result = paginator.paginate(Bucket=self.bucket_name,
                                    Prefix=self.prefix,
                                    Delimiter='/')
        classes = []
        for page in result:
            for cp in page.get('CommonPrefixes', []):
                # Extract class name from prefix.
                class_name = cp['Prefix'].replace(self.prefix, '').strip('/')
                if class_name:
                    classes.append(class_name)
        classes.sort()
        class_to_idx = {cls: idx for idx, cls in enumerate(classes)}
        return classes, class_to_idx

    def _make_dataset(self):
        """
        Build a list of (s3_key, class_index) for all images under each class folder.
        """
        samples = []
        for cls in self.classes:
            cls_prefix = os.path.join(self.prefix, cls) + '/'
            paginator = self.s3_client.get_paginator('list_objects_v2')
            pages = paginator.paginate(Bucket=self.bucket_name, Prefix=cls_prefix)
            for page in pages:
                for obj in page.get('Contents', []):
                    key = obj['Key']
                    # Accept only files with standard image extensions.
                    if key.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif')):
                        samples.append((key, self.class_to_idx[cls]))
        return samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        key, label = self.samples[index]
        # Get the object from S3 as bytes.
        response = self.s3_client.get_object(Bucket=self.bucket_name, Key=key)
        img_bytes = response['Body'].read()
        image = Image.open(io.BytesIO(img_bytes)).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label

In [ ]:
# ----- Data Transforms -----
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(192),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize(192),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

test_transforms = transforms.Compose([
    transforms.Resize(192),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

In [ ]:
# For swinv2_base_window12_192_22k, the expected image size is 192.
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(192),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_transforms = transforms.Compose([
    transforms.Resize(192),
    transforms.CenterCrop(192),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

test_transforms = transforms.Compose([
    transforms.Resize(192),
    transforms.CenterCrop(192),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

In [ ]:
# ----- S3 Bucket and Prefix Settings -----
bucket_name = 'students-projects'
# Prefix to your training images stored in S3.
train_prefix = 'Data/train-20250409T041124Z-001/train/'
# Prefix for validation images in S3 (adjust if needed).
val_prefix = 'Data/val-20250409T041716Z-001/val/'
# Prefix for validation images in S3 (adjust if needed).
test_prefix = 'Data/test-20250409T041654Z-001/test/'

In [ ]:
# ----- Initialize Datasets -----
train_dataset = S3ImageFolder(bucket_name, train_prefix, transform=train_transforms)
val_dataset = S3ImageFolder(bucket_name, val_prefix, transform=val_transforms)
test_dataset = S3ImageFolder(bucket_name, test_prefix, transform=test_transforms)

In [ ]:
from torch.utils.data import ConcatDataset

# Merge train and validation datasets
train_dataset = ConcatDataset([train_dataset, val_dataset])

print("Total length of merged train_dataset:", len(train_dataset))

print("Total length of test_dataset:", len(test_dataset))

Total length of merged train_dataset: 1465
Total length of test_dataset: 137


In [ ]:
BATCH_SIZE = 32
NUM_WORKERS = 4

# Create DataLoader for the merged dataset
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, pin_memory=True)

In [ ]:
# Get the total length of the merged dataset
train_samples = len(train_loader.dataset)


# Print the total number of samples in the merged dataset
print(f"Total samples in train_loader: {train_samples}")

Total samples in train_loader: 1465


In [ ]:
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=True,
                        num_workers=NUM_WORKERS, pin_memory=True)

In [ ]:
for i in test_loader:
    print(i)

[tensor([[[[ 2.1290,  2.1804,  1.8893,  ...,  1.7352,  1.3755,  1.8379],
          [ 1.9749,  2.0263,  1.4783,  ...,  1.6838,  1.9235,  2.0777],
          [ 2.0092,  2.1119,  1.7523,  ...,  2.0092,  2.1290,  2.0605],
          ...,
          [ 1.9064,  2.1633,  2.1290,  ...,  1.8379,  1.5639,  1.9920],
          [ 1.7352,  1.8550,  2.1119,  ...,  1.7865,  1.8893,  1.9749],
          [ 1.9920,  1.8379,  2.1804,  ...,  1.9407,  1.8379,  1.9578]],

         [[ 2.0259,  1.9734,  0.6779,  ...,  0.2402, -0.1099,  0.2752],
          [ 1.4832,  1.5532, -0.3550,  ...,  0.0651,  0.3627,  0.6254],
          [ 1.0805,  1.8333,  0.1877,  ...,  0.4678,  0.7304,  0.8004],
          ...,
          [ 0.4853,  0.5728,  0.8179,  ..., -0.0224, -0.3725,  0.4503],
          [ 0.5378,  0.6604,  1.1856,  ...,  0.4853,  0.0826,  0.7304],
          [ 1.5357,  1.4132,  1.9909,  ...,  0.9230,  0.1001,  0.8004]],

         [[ 2.4483,  2.4657,  2.0125,  ...,  2.0125,  1.7860,  2.0125],
          [ 2.2740,  2.3088, 

In [ ]:
val_loader=test_loader

In [ ]:
# Get the total length of the merged dataset
train_samples = len(val_loader.dataset)


# Print the total number of samples in the merged dataset
print(f"Total samples in train_loader: {train_samples}")

Total samples in train_loader: 137


In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import timm
from torch.optim.lr_scheduler import CosineAnnealingLR
import optuna

# --- Settings ---
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.backends.cudnn.benchmark = True
BATCH_SIZE = 32
NUM_CLASSES = 3         # Set to your number of classes
NUM_EPOCHS = 30
NUM_WORKERS = 4
PATIENCE = 5            # Early stopping patience




# --- Training Function with Best Model Saving ---
def train_model(model, optimizer, criterion, scheduler, save_path):
    best_acc = 0
    best_state = None
    trigger_times = 0

    for epoch in range(NUM_EPOCHS):
        model.train()
        total_train_loss, correct_train, total_train = 0, 0, 0

        for inputs, targets in train_loader:
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()

            total_train_loss += loss.item()
            _, predicted = outputs.max(1)
            total_train += targets.size(0)
            correct_train += predicted.eq(targets).sum().item()

        scheduler.step()
        train_acc = 100. * correct_train / total_train

        # --- Validation ---
        model.eval()
        correct_val, total_val = 0, 0
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
                outputs = model(inputs)
                _, predicted = outputs.max(1)
                total_val += targets.size(0)
                correct_val += predicted.eq(targets).sum().item()

        val_acc = 100. * correct_val / total_val
        print(f'Epoch [{epoch+1}/{NUM_EPOCHS}] Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%')

        # --- Save Best Model ---
        if val_acc > best_acc:
            best_acc = val_acc
            # For DataParallel use model.module.state_dict(), otherwise model.state_dict()
            best_state = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
            torch.save(best_state, save_path)
            print(f"Best model updated (Val Acc: {best_acc:.2f}%) and saved to {save_path}")
            trigger_times = 0
        else:
            trigger_times += 1
            if trigger_times >= PATIENCE:
                print("Early stopping triggered!")
                break

    return train_acc, best_acc

# --- Optuna Hyperparameter Objective for convnextv2 ---
def objective(trial):
    model_name = "convnextv2_base"
    model = timm.create_model(model_name, pretrained=True, num_classes=NUM_CLASSES)
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
    model.to(DEVICE)

    optimizer_name = trial.suggest_categorical('optimizer', ['AdamW', 'Lion'])
    lr = trial.suggest_float('lr', 1e-5, 1e-3, log=True)
    weight_decay = trial.suggest_float('weight_decay', 1e-6, 1e-2, log=True)

    if optimizer_name == 'AdamW':
        optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    else:
        from lion_pytorch import Lion
        optimizer = Lion(model.parameters(), lr=lr, weight_decay=weight_decay)

    scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
    label_smoothing = trial.suggest_float('label_smoothing', 0.0, 0.2)
    criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)

    # Save the best model from this trial to "best.pt"
    best_model_path = "convnextv2_base.pt"
    train_acc, val_acc = train_model(model, optimizer, criterion, scheduler, save_path=best_model_path)
    return val_acc

# --- Run Optuna Hyperparameter Optimization ---
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=10)

print("Best trial:")
best_trial = study.best_trial
print(f"  Value: {best_trial.value}")
print("  Params: ")
for key, value in best_trial.params.items():
    print(f"    {key}: {value}")

# --- After Optimization, you can load your best model ---
best_model_state = torch.load("convnextv2_base.pt", map_location=DEVICE)
model = timm.create_model("convnextv2_base", pretrained=False, num_classes=NUM_CLASSES)
model.to(DEVICE)
model.load_state_dict(best_model_state)
model.eval()
print("Best model loaded successfully for inference!")


[I 2025-04-11 07:52:47,460] A new study created in memory with name: no-name-86eb53e1-cba5-4f30-bda3-2afc49443596


Epoch [1/30] Train Acc: 85.67% | Val Acc: 96.35%
Best model updated (Val Acc: 96.35%) and saved to convnextv2_base.pt
Epoch [2/30] Train Acc: 97.68% | Val Acc: 97.08%
Best model updated (Val Acc: 97.08%) and saved to convnextv2_base.pt
Epoch [3/30] Train Acc: 98.02% | Val Acc: 91.97%
Epoch [4/30] Train Acc: 98.70% | Val Acc: 97.08%
Epoch [5/30] Train Acc: 99.39% | Val Acc: 98.54%
Best model updated (Val Acc: 98.54%) and saved to convnextv2_base.pt
Epoch [6/30] Train Acc: 98.09% | Val Acc: 97.81%
Epoch [7/30] Train Acc: 99.59% | Val Acc: 98.54%
Epoch [8/30] Train Acc: 99.59% | Val Acc: 98.54%
Epoch [9/30] Train Acc: 99.73% | Val Acc: 98.54%
Epoch [10/30] Train Acc: 99.18% | Val Acc: 99.27%
Best model updated (Val Acc: 99.27%) and saved to convnextv2_base.pt
Epoch [11/30] Train Acc: 99.11% | Val Acc: 99.27%
Epoch [12/30] Train Acc: 99.73% | Val Acc: 98.54%
Epoch [13/30] Train Acc: 99.93% | Val Acc: 99.27%
Epoch [14/30] Train Acc: 99.73% | Val Acc: 99.27%


[I 2025-04-11 08:12:48,366] Trial 0 finished with value: 99.27007299270073 and parameters: {'optimizer': 'AdamW', 'lr': 1.5415737949808744e-05, 'weight_decay': 0.00013394842344726645, 'label_smoothing': 0.10261801192515618}. Best is trial 0 with value: 99.27007299270073.


Epoch [15/30] Train Acc: 99.39% | Val Acc: 99.27%
Early stopping triggered!
Epoch [1/30] Train Acc: 86.01% | Val Acc: 95.62%
Best model updated (Val Acc: 95.62%) and saved to convnextv2_base.pt
Epoch [2/30] Train Acc: 96.45% | Val Acc: 95.62%
Epoch [3/30] Train Acc: 96.72% | Val Acc: 99.27%
Best model updated (Val Acc: 99.27%) and saved to convnextv2_base.pt
Epoch [4/30] Train Acc: 98.43% | Val Acc: 99.27%
Epoch [5/30] Train Acc: 98.43% | Val Acc: 96.35%
Epoch [6/30] Train Acc: 98.02% | Val Acc: 99.27%
Epoch [7/30] Train Acc: 98.63% | Val Acc: 98.54%
Epoch [8/30] Train Acc: 98.98% | Val Acc: 100.00%
Best model updated (Val Acc: 100.00%) and saved to convnextv2_base.pt
Epoch [9/30] Train Acc: 98.84% | Val Acc: 100.00%
Epoch [10/30] Train Acc: 98.70% | Val Acc: 100.00%
Epoch [11/30] Train Acc: 98.84% | Val Acc: 97.81%
Epoch [12/30] Train Acc: 99.66% | Val Acc: 97.81%


[I 2025-04-11 08:29:47,048] Trial 1 finished with value: 100.0 and parameters: {'optimizer': 'Lion', 'lr': 2.513112196049506e-05, 'weight_decay': 0.0007870298800475523, 'label_smoothing': 0.005885279370371133}. Best is trial 1 with value: 100.0.


...

Epoch [9/30] Train Acc: 36.25% | Val Acc: 68.61%
Early stopping triggered!
Epoch [1/30] Train Acc: 78.98% | Val Acc: 90.51%
Best model updated (Val Acc: 90.51%) and saved to convnextv2_base.pt
Epoch [2/30] Train Acc: 95.22% | Val Acc: 98.54%
Best model updated (Val Acc: 98.54%) and saved to convnextv2_base.pt
Epoch [3/30] Train Acc: 96.25% | Val Acc: 98.54%
Epoch [4/30] Train Acc: 96.79% | Val Acc: 94.89%
Epoch [5/30] Train Acc: 96.66% | Val Acc: 86.86%
Epoch [6/30] Train Acc: 96.93% | Val Acc: 94.89%


[I 2025-04-11 09:42:30,278] Trial 8 finished with value: 98.54014598540147 and parameters: {'optimizer': 'Lion', 'lr': 5.982814727767745e-05, 'weight_decay': 4.381257399440973e-06, 'label_smoothing': 0.0855335760959713}. Best is trial 1 with value: 100.0.


Epoch [7/30] Train Acc: 97.68% | Val Acc: 94.16%
Early stopping triggered!
Epoch [1/30] Train Acc: 34.54% | Val Acc: 20.44%
Best model updated (Val Acc: 20.44%) and saved to convnextv2_base.pt
Epoch [2/30] Train Acc: 34.68% | Val Acc: 68.61%
Best model updated (Val Acc: 68.61%) and saved to convnextv2_base.pt
Epoch [3/30] Train Acc: 36.11% | Val Acc: 68.61%
Epoch [4/30] Train Acc: 36.04% | Val Acc: 10.95%
Epoch [5/30] Train Acc: 35.56% | Val Acc: 68.61%
Epoch [6/30] Train Acc: 35.90% | Val Acc: 68.61%


[I 2025-04-11 09:51:34,872] Trial 9 finished with value: 68.61313868613139 and parameters: {'optimizer': 'Lion', 'lr': 0.00012115737121207203, 'weight_decay': 0.00464437842490229, 'label_smoothing': 0.05741942621651302}. Best is trial 1 with value: 100.0.


Epoch [7/30] Train Acc: 35.43% | Val Acc: 68.61%
Early stopping triggered!
Best trial:
  Value: 100.0
  Params: 
    optimizer: Lion
    lr: 2.513112196049506e-05
    weight_decay: 0.0007870298800475523
    label_smoothing: 0.005885279370371133
Best model loaded successfully for inference!


In [ ]:
    import os
    import torch
    import torch.nn as nn
    import torch.optim as optim
    import torchvision
    import torchvision.transforms as transforms
    from torch.utils.data import DataLoader
    import timm
    from torch.optim.lr_scheduler import CosineAnnealingLR
    import optuna

    # --- Settings ---
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    torch.backends.cudnn.benchmark = True
    BATCH_SIZE = 32
    NUM_CLASSES = 3         # Set to your number of classes
    NUM_EPOCHS = 30
    NUM_WORKERS = 4
    PATIENCE = 5            # Early stopping patience




    # --- Training Function with Best Model Saving ---
    def train_model(model, optimizer, criterion, scheduler, save_path):
        best_acc = 0
        best_state = None
        trigger_times = 0

        for epoch in range(NUM_EPOCHS):
            model.train()
            total_train_loss, correct_train, total_train = 0, 0, 0

            for inputs, targets in train_loader:
                inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
                optimizer.zero_grad()
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                loss.backward()
                optimizer.step()

                total_train_loss += loss.item()
                _, predicted = outputs.max(1)
                total_train += targets.size(0)
                correct_train += predicted.eq(targets).sum().item()

            scheduler.step()
            train_acc = 100. * correct_train / total_train

            # --- Validation ---
            model.eval()
            correct_val, total_val = 0, 0
            with torch.no_grad():
                for inputs, targets in val_loader:
                    inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
                    outputs = model(inputs)
                    _, predicted = outputs.max(1)
                    total_val += targets.size(0)
                    correct_val += predicted.eq(targets).sum().item()

            val_acc = 100. * correct_val / total_val
            print(f'Epoch [{epoch+1}/{NUM_EPOCHS}] Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%')

            # --- Save Best Model ---
            if val_acc > best_acc:
                best_acc = val_acc
                # For DataParallel use model.module.state_dict(), otherwise model.state_dict()
                best_state = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
                torch.save(best_state, save_path)
                print(f"Best model updated (Val Acc: {best_acc:.2f}%) and saved to {save_path}")
                trigger_times = 0
            else:
                trigger_times += 1
                if trigger_times >= PATIENCE:
                    print("Early stopping triggered!")
                    break

        return train_acc, best_acc

    # --- Optuna Hyperparameter Objective for efficientvit_b2 ---
    def objective(trial):
        model_name = "efficientvit_b2"
        model = timm.create_model(model_name, pretrained=True, num_classes=NUM_CLASSES)
        if torch.cuda.device_count() > 1:
            model = nn.DataParallel(model)
        model.to(DEVICE)

        optimizer_name = trial.suggest_categorical('optimizer', ['AdamW', 'Lion'])
        lr = trial.suggest_float('lr', 1e-5, 1e-3, log=True)
        weight_decay = trial.suggest_float('weight_decay', 1e-6, 1e-2, log=True)

        if optimizer_name == 'AdamW':
            optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
        else:
            from lion_pytorch import Lion
            optimizer = Lion(model.parameters(), lr=lr, weight_decay=weight_decay)

        scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
        label_smoothing = trial.suggest_float('label_smoothing', 0.0, 0.2)
        criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)

        # Save the best model from this trial to "best.pt"
        best_model_path = "efficientvit_b2.pt"
        train_acc, val_acc = train_model(model, optimizer, criterion, scheduler, save_path=best_model_path)
        return val_acc

    # --- Run Optuna Hyperparameter Optimization ---
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=10)

    print("Best trial:")
    best_trial = study.best_trial
    print(f"  Value: {best_trial.value}")
    print("  Params: ")
    for key, value in best_trial.params.items():
        print(f"    {key}: {value}")

    # --- After Optimization, you can load your best model ---
    best_model_state = torch.load("efficientvit_b2.pt", map_location=DEVICE)
    model = timm.create_model("efficientvit_b2", pretrained=False, num_classes=NUM_CLASSES)
    model.to(DEVICE)
    model.load_state_dict(best_model_state)
    model.eval()
    print("Best model loaded successfully for inference!")


[I 2025-04-11 10:05:19,396] A new study created in memory with name: no-name-84476a1b-d776-4f2a-9122-ddc6894c0d47


Epoch [1/30] Train Acc: 80.89% | Val Acc: 10.95%
Best model updated (Val Acc: 10.95%) and saved to efficientvit_b2.pt
Epoch [2/30] Train Acc: 89.08% | Val Acc: 11.68%
Best model updated (Val Acc: 11.68%) and saved to efficientvit_b2.pt
Epoch [3/30] Train Acc: 90.03% | Val Acc: 31.39%
Best model updated (Val Acc: 31.39%) and saved to efficientvit_b2.pt
Epoch [4/30] Train Acc: 92.22% | Val Acc: 32.85%
Best model updated (Val Acc: 32.85%) and saved to efficientvit_b2.pt
Epoch [5/30] Train Acc: 94.61% | Val Acc: 71.53%
Best model updated (Val Acc: 71.53%) and saved to efficientvit_b2.pt
Epoch [6/30] Train Acc: 94.33% | Val Acc: 62.04%
Epoch [7/30] Train Acc: 96.25% | Val Acc: 97.81%
Best model updated (Val Acc: 97.81%) and saved to efficientvit_b2.pt
Epoch [8/30] Train Acc: 95.56% | Val Acc: 76.64%
Epoch [9/30] Train Acc: 95.09% | Val Acc: 86.86%
Epoch [10/30] Train Acc: 96.31% | Val Acc: 67.88%
Epoch [11/30] Train Acc: 95.22% | Val Acc: 81.02%


[I 2025-04-11 10:14:40,696] Trial 0 finished with value: 97.81021897810218 and parameters: {'optimizer': 'Lion', 'lr': 0.0007084006547253661, 'weight_decay': 0.0001909909076687618, 'label_smoothing': 0.015889203204655367}. Best is trial 0 with value: 97.81021897810218.


Epoch [12/30] Train Acc: 97.06% | Val Acc: 96.35%
Early stopping triggered!
Epoch [1/30] Train Acc: 76.45% | Val Acc: 89.78%
Best model updated (Val Acc: 89.78%) and saved to efficientvit_b2.pt
Epoch [2/30] Train Acc: 94.40% | Val Acc: 97.81%
Best model updated (Val Acc: 97.81%) and saved to efficientvit_b2.pt
Epoch [3/30] Train Acc: 98.23% | Val Acc: 99.27%
Best model updated (Val Acc: 99.27%) and saved to efficientvit_b2.pt
Epoch [4/30] Train Acc: 98.63% | Val Acc: 98.54%
Epoch [5/30] Train Acc: 99.39% | Val Acc: 100.00%
Best model updated (Val Acc: 100.00%) and saved to efficientvit_b2.pt
Epoch [6/30] Train Acc: 99.11% | Val Acc: 99.27%
Epoch [7/30] Train Acc: 98.57% | Val Acc: 99.27%
Epoch [8/30] Train Acc: 99.32% | Val Acc: 100.00%
Epoch [9/30] Train Acc: 98.91% | Val Acc: 99.27%


[I 2025-04-11 10:21:08,388] Trial 1 finished with value: 100.0 and parameters: {'optimizer': 'Lion', 'lr': 3.059415603423081e-05, 'weight_decay': 1.1608832776865003e-06, 'label_smoothing': 0.048191160814897854}. Best is trial 1 with value: 100.0.


...

Epoch [13/30] Train Acc: 99.73% | Val Acc: 100.00%
Early stopping triggered!
Epoch [1/30] Train Acc: 87.30% | Val Acc: 75.91%
Best model updated (Val Acc: 75.91%) and saved to efficientvit_b2.pt
Epoch [2/30] Train Acc: 96.31% | Val Acc: 92.70%
Best model updated (Val Acc: 92.70%) and saved to efficientvit_b2.pt
Epoch [3/30] Train Acc: 98.43% | Val Acc: 97.08%
Best model updated (Val Acc: 97.08%) and saved to efficientvit_b2.pt
Epoch [4/30] Train Acc: 97.75% | Val Acc: 99.27%
Best model updated (Val Acc: 99.27%) and saved to efficientvit_b2.pt
Epoch [5/30] Train Acc: 98.02% | Val Acc: 99.27%
Epoch [6/30] Train Acc: 98.91% | Val Acc: 95.62%
Epoch [7/30] Train Acc: 99.52% | Val Acc: 100.00%
Best model updated (Val Acc: 100.00%) and saved to efficientvit_b2.pt
Epoch [8/30] Train Acc: 99.32% | Val Acc: 100.00%
Epoch [9/30] Train Acc: 99.39% | Val Acc: 100.00%
Epoch [10/30] Train Acc: 99.45% | Val Acc: 99.27%
Epoch [11/30] Train Acc: 99.52% | Val Acc: 100.00%


[I 2025-04-11 11:14:09,408] Trial 8 finished with value: 100.0 and parameters: {'optimizer': 'AdamW', 'lr': 0.0002773521306788028, 'weight_decay': 0.002369477789259127, 'label_smoothing': 0.03393724869354977}. Best is trial 1 with value: 100.0.


Epoch [12/30] Train Acc: 99.59% | Val Acc: 98.54%
Early stopping triggered!
Epoch [1/30] Train Acc: 72.42% | Val Acc: 76.64%
Best model updated (Val Acc: 76.64%) and saved to efficientvit_b2.pt
Epoch [2/30] Train Acc: 91.26% | Val Acc: 91.24%
Best model updated (Val Acc: 91.24%) and saved to efficientvit_b2.pt
Epoch [3/30] Train Acc: 93.92% | Val Acc: 94.16%
Best model updated (Val Acc: 94.16%) and saved to efficientvit_b2.pt
Epoch [4/30] Train Acc: 97.20% | Val Acc: 97.81%
Best model updated (Val Acc: 97.81%) and saved to efficientvit_b2.pt
Epoch [5/30] Train Acc: 97.20% | Val Acc: 97.81%
Epoch [6/30] Train Acc: 98.50% | Val Acc: 99.27%
Best model updated (Val Acc: 99.27%) and saved to efficientvit_b2.pt
Epoch [7/30] Train Acc: 98.43% | Val Acc: 99.27%
Epoch [8/30] Train Acc: 98.91% | Val Acc: 99.27%
Epoch [9/30] Train Acc: 98.98% | Val Acc: 99.27%
Epoch [10/30] Train Acc: 98.77% | Val Acc: 99.27%
Epoch [11/30] Train Acc: 99.04% | Val Acc: 100.00%
Best model updated (Val Acc: 100.00%)

[I 2025-04-11 11:24:09,227] Trial 9 finished with value: 100.0 and parameters: {'optimizer': 'AdamW', 'lr': 2.9590792931230234e-05, 'weight_decay': 0.005130325350835134, 'label_smoothing': 0.0749138763837222}. Best is trial 1 with value: 100.0.


Epoch [16/30] Train Acc: 99.32% | Val Acc: 100.00%
Early stopping triggered!
Best trial:
  Value: 100.0
  Params: 
    optimizer: Lion
    lr: 3.059415603423081e-05
    weight_decay: 1.1608832776865003e-06
    label_smoothing: 0.048191160814897854
Best model loaded successfully for inference!


In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import timm
from torch.optim.lr_scheduler import CosineAnnealingLR
import optuna

# --- Settings ---
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.backends.cudnn.benchmark = True
BATCH_SIZE = 32
NUM_CLASSES = 3         # Set to your number of classes
NUM_EPOCHS = 30
NUM_WORKERS = 4
PATIENCE = 5            # Early stopping patience




# --- Training Function with Best Model Saving ---
def train_model(model, optimizer, criterion, scheduler, save_path):
    best_acc = 0
    best_state = None
    trigger_times = 0

    for epoch in range(NUM_EPOCHS):
        model.train()
        total_train_loss, correct_train, total_train = 0, 0, 0

        for inputs, targets in train_loader:
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()

            total_train_loss += loss.item()
            _, predicted = outputs.max(1)
            total_train += targets.size(0)
            correct_train += predicted.eq(targets).sum().item()

        scheduler.step()
        train_acc = 100. * correct_train / total_train

        # --- Validation ---
        model.eval()
        correct_val, total_val = 0, 0
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
                outputs = model(inputs)
                _, predicted = outputs.max(1)
                total_val += targets.size(0)
                correct_val += predicted.eq(targets).sum().item()

        val_acc = 100. * correct_val / total_val
        print(f'Epoch [{epoch+1}/{NUM_EPOCHS}] Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%')

        # --- Save Best Model ---
        if val_acc > best_acc:
            best_acc = val_acc
            # For DataParallel use model.module.state_dict(), otherwise model.state_dict()
            best_state = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
            torch.save(best_state, save_path)
            print(f"Best model updated (Val Acc: {best_acc:.2f}%) and saved to {save_path}")
            trigger_times = 0
        else:
            trigger_times += 1
            if trigger_times >= PATIENCE:
                print("Early stopping triggered!")
                break

    return train_acc, best_acc

# --- Optuna Hyperparameter Objective for swinv2_base_window12_192_22k ---
def objective(trial):
    model_name = "swinv2_base_window12_192_22k"
    model = timm.create_model(model_name, pretrained=True, num_classes=NUM_CLASSES)
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
    model.to(DEVICE)

    optimizer_name = trial.suggest_categorical('optimizer', ['AdamW', 'Lion'])
    lr = trial.suggest_float('lr', 1e-5, 1e-3, log=True)
    weight_decay = trial.suggest_float('weight_decay', 1e-6, 1e-2, log=True)

    if optimizer_name == 'AdamW':
        optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    else:
        from lion_pytorch import Lion
        optimizer = Lion(model.parameters(), lr=lr, weight_decay=weight_decay)

    scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
    label_smoothing = trial.suggest_float('label_smoothing', 0.0, 0.2)
    criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)

    # Save the best model from this trial to "best.pt"
    best_model_path = "swinv2_base_window12_192_22k.pt"
    train_acc, val_acc = train_model(model, optimizer, criterion, scheduler, save_path=best_model_path)
    return val_acc

# --- Run Optuna Hyperparameter Optimization ---
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=10)

print("Best trial:")
best_trial = study.best_trial
print(f"  Value: {best_trial.value}")
print("  Params: ")
for key, value in best_trial.params.items():
    print(f"    {key}: {value}")

# --- After Optimization, you can load your best model ---
best_model_state = torch.load("swinv2_base_window12_192_22k.pt", map_location=DEVICE)
model = timm.create_model("swinv2_base_window12_192_22k", pretrained=False, num_classes=NUM_CLASSES)
model.to(DEVICE)
model.load_state_dict(best_model_state)
model.eval()
print("Best model loaded successfully for inference!")


[I 2025-04-14 05:46:09,941] A new study created in memory with name: no-name-c8cdf6ae-8ef2-4a22-9bcd-3115e0b28704
/home/guser6/my_env/lib/python3.10/site-packages/timm/models/_factory.py:126: UserWarning: Mapping deprecated model name swinv2_base_window12_192_22k to current swinv2_base_window12_192.ms_in22k.
  model = create_fn(


Epoch [1/30] Train Acc: 86.14% | Val Acc: 85.40%
Best model updated (Val Acc: 85.40%) and saved to swinv2_base_window12_192_22k.pt
Epoch [2/30] Train Acc: 96.93% | Val Acc: 99.27%
Best model updated (Val Acc: 99.27%) and saved to swinv2_base_window12_192_22k.pt
Epoch [3/30] Train Acc: 98.50% | Val Acc: 96.35%
Epoch [4/30] Train Acc: 98.50% | Val Acc: 99.27%
Epoch [5/30] Train Acc: 98.09% | Val Acc: 99.27%
Epoch [6/30] Train Acc: 98.91% | Val Acc: 98.54%


[I 2025-04-14 05:52:07,634] Trial 0 finished with value: 99.27007299270073 and parameters: {'optimizer': 'AdamW', 'lr': 1.8114797425704468e-05, 'weight_decay': 1.2847851938424106e-06, 'label_smoothing': 0.16228153973075485}. Best is trial 0 with value: 99.27007299270073.


Epoch [7/30] Train Acc: 99.25% | Val Acc: 99.27%
Early stopping triggered!
Epoch [1/30] Train Acc: 41.16% | Val Acc: 42.34%
Best model updated (Val Acc: 42.34%) and saved to swinv2_base_window12_192_22k.pt
Epoch [2/30] Train Acc: 60.00% | Val Acc: 44.53%
Best model updated (Val Acc: 44.53%) and saved to swinv2_base_window12_192_22k.pt
Epoch [3/30] Train Acc: 65.32% | Val Acc: 48.18%
Best model updated (Val Acc: 48.18%) and saved to swinv2_base_window12_192_22k.pt
Epoch [4/30] Train Acc: 66.83% | Val Acc: 37.23%
Epoch [5/30] Train Acc: 67.37% | Val Acc: 70.07%
Best model updated (Val Acc: 70.07%) and saved to swinv2_base_window12_192_22k.pt
Epoch [6/30] Train Acc: 69.08% | Val Acc: 56.20%
Epoch [7/30] Train Acc: 69.28% | Val Acc: 45.99%
Epoch [8/30] Train Acc: 73.17% | Val Acc: 64.23%
Epoch [9/30] Train Acc: 79.39% | Val Acc: 73.72%
Best model updated (Val Acc: 73.72%) and saved to swinv2_base_window12_192_22k.pt
Epoch [10/30] Train Acc: 78.36% | Val Acc: 67.88%
Epoch [11/30] Train Acc:

[I 2025-04-14 06:08:39,744] Trial 1 finished with value: 95.62043795620438 and parameters: {'optimizer': 'AdamW', 'lr': 0.0005932320982963075, 'weight_decay': 8.658971295664411e-06, 'label_smoothing': 0.04214087033881624}. Best is trial 0 with value: 99.27007299270073.


...

Epoch [7/30] Train Acc: 33.65% | Val Acc: 68.61%
Early stopping triggered!
Epoch [1/30] Train Acc: 43.62% | Val Acc: 32.85%
Best model updated (Val Acc: 32.85%) and saved to swinv2_base_window12_192_22k.pt
Epoch [2/30] Train Acc: 62.12% | Val Acc: 44.53%
Best model updated (Val Acc: 44.53%) and saved to swinv2_base_window12_192_22k.pt
Epoch [3/30] Train Acc: 69.15% | Val Acc: 60.58%
Best model updated (Val Acc: 60.58%) and saved to swinv2_base_window12_192_22k.pt
Epoch [4/30] Train Acc: 76.25% | Val Acc: 46.72%
Epoch [5/30] Train Acc: 73.86% | Val Acc: 79.56%
Best model updated (Val Acc: 79.56%) and saved to swinv2_base_window12_192_22k.pt
Epoch [6/30] Train Acc: 84.44% | Val Acc: 89.78%
Best model updated (Val Acc: 89.78%) and saved to swinv2_base_window12_192_22k.pt
Epoch [7/30] Train Acc: 89.15% | Val Acc: 84.67%
Epoch [8/30] Train Acc: 89.49% | Val Acc: 94.89%
Best model updated (Val Acc: 94.89%) and saved to swinv2_base_window12_192_22k.pt
Epoch [9/30] Train Acc: 93.52% | Val Acc:

[I 2025-04-14 07:05:38,000] Trial 8 finished with value: 100.0 and parameters: {'optimizer': 'AdamW', 'lr': 0.00036250196737959173, 'weight_decay': 0.005998427691523543, 'label_smoothing': 0.09564506943496416}. Best is trial 8 with value: 100.0.


Epoch [21/30] Train Acc: 99.80% | Val Acc: 100.00%
Early stopping triggered!
Epoch [1/30] Train Acc: 83.14% | Val Acc: 89.78%
Best model updated (Val Acc: 89.78%) and saved to swinv2_base_window12_192_22k.pt
Epoch [2/30] Train Acc: 94.20% | Val Acc: 91.24%
Best model updated (Val Acc: 91.24%) and saved to swinv2_base_window12_192_22k.pt
Epoch [3/30] Train Acc: 97.13% | Val Acc: 97.81%
Best model updated (Val Acc: 97.81%) and saved to swinv2_base_window12_192_22k.pt
Epoch [4/30] Train Acc: 97.13% | Val Acc: 97.81%
Epoch [5/30] Train Acc: 97.61% | Val Acc: 94.16%
Epoch [6/30] Train Acc: 98.57% | Val Acc: 99.27%
Best model updated (Val Acc: 99.27%) and saved to swinv2_base_window12_192_22k.pt
Epoch [7/30] Train Acc: 97.41% | Val Acc: 94.16%
Epoch [8/30] Train Acc: 98.23% | Val Acc: 97.08%
Epoch [9/30] Train Acc: 99.11% | Val Acc: 94.89%
Epoch [10/30] Train Acc: 98.77% | Val Acc: 95.62%
Epoch [11/30] Train Acc: 98.36% | Val Acc: 100.00%
Best model updated (Val Acc: 100.00%) and saved to sw

[I 2025-04-14 07:18:59,123] Trial 9 finished with value: 100.0 and parameters: {'optimizer': 'AdamW', 'lr': 7.705805102930411e-05, 'weight_decay': 1.3540324495611982e-05, 'label_smoothing': 0.09961708659988595}. Best is trial 8 with value: 100.0.


Epoch [16/30] Train Acc: 99.73% | Val Acc: 100.00%
Early stopping triggered!
Best trial:
  Value: 100.0
  Params: 
    optimizer: AdamW
    lr: 0.00036250196737959173
    weight_decay: 0.005998427691523543
    label_smoothing: 0.09564506943496416
Best model loaded successfully for inference!


In [ ]:
import os
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

# === Constants ===
IMAGE_SIZE = 192
BATCH_SIZE = 32
NUM_WORKERS = 4

# === Transforms ===
train_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),  # Ensure uniform size
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)
])

val_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),  # Same for validation
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)
])

# === Datasets ===
train_dataset = ImageFolder(root='path/to/train', transform=train_transforms)
val_dataset = ImageFolder(root='path/to/val', transform=val_transforms)

# === DataLoaders ===
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)


In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import timm
from torch.optim.lr_scheduler import CosineAnnealingLR
import optuna

# --- Settings ---
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.backends.cudnn.benchmark = True
BATCH_SIZE = 32
NUM_CLASSES = 3         # Set to your number of classes
NUM_EPOCHS = 30
NUM_WORKERS = 4
PATIENCE = 5            # Early stopping patience




# --- Training Function with Best Model Saving ---
def train_model(model, optimizer, criterion, scheduler, save_path):
    best_acc = 0
    best_state = None
    trigger_times = 0

    for epoch in range(NUM_EPOCHS):
        model.train()
        total_train_loss, correct_train, total_train = 0, 0, 0

        for inputs, targets in train_loader:
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()

            total_train_loss += loss.item()
            _, predicted = outputs.max(1)
            total_train += targets.size(0)
            correct_train += predicted.eq(targets).sum().item()

        scheduler.step()
        train_acc = 100. * correct_train / total_train

        # --- Validation ---
        model.eval()
        correct_val, total_val = 0, 0
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
                outputs = model(inputs)
                _, predicted = outputs.max(1)
                total_val += targets.size(0)
                correct_val += predicted.eq(targets).sum().item()

        val_acc = 100. * correct_val / total_val
        print(f'Epoch [{epoch+1}/{NUM_EPOCHS}] Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%')

        # --- Save Best Model ---
        if val_acc > best_acc:
            best_acc = val_acc
            # For DataParallel use model.module.state_dict(), otherwise model.state_dict()
            best_state = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
            torch.save(best_state, save_path)
            print(f"Best model updated (Val Acc: {best_acc:.2f}%) and saved to {save_path}")
            trigger_times = 0
        else:
            trigger_times += 1
            if trigger_times >= PATIENCE:
                print("Early stopping triggered!")
                break

    return train_acc, best_acc

# --- Optuna Hyperparameter Objective for efficientvit_b2 ---
def objective(trial):
    model_name = "swinv2_base_window12_192_22k"
    model = timm.create_model(model_name, pretrained=True, num_classes=NUM_CLASSES)
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
    model.to(DEVICE)

    optimizer_name = trial.suggest_categorical('optimizer', ['AdamW', 'Lion'])
    lr = trial.suggest_float('lr', 1e-5, 1e-3, log=True)
    weight_decay = trial.suggest_float('weight_decay', 1e-6, 1e-2, log=True)

    if optimizer_name == 'AdamW':
        optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    else:
        from lion_pytorch import Lion
        optimizer = Lion(model.parameters(), lr=lr, weight_decay=weight_decay)

    scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
    label_smoothing = trial.suggest_float('label_smoothing', 0.0, 0.2)
    criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)

    # Save the best model from this trial to "best.pt"
    best_model_path = "swinv2_base_window12_192_22k.pt"
    train_acc, val_acc = train_model(model, optimizer, criterion, scheduler, save_path=best_model_path)
    return val_acc

# --- Run Optuna Hyperparameter Optimization ---
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=10)

print("Best trial:")
best_trial = study.best_trial
print(f"  Value: {best_trial.value}")
print("  Params: ")
for key, value in best_trial.params.items():
    print(f"    {key}: {value}")

# --- After Optimization, you can load your best model ---
best_model_state = torch.load("swinv2_base_window12_192_22k.pt", map_location=DEVICE)
model = timm.create_model("swinv2_base_window12_192_22k", pretrained=False, num_classes=NUM_CLASSES)
model.to(DEVICE)
model.load_state_dict(best_model_state)
model.eval()
print("Best model loaded successfully for inference!")


In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import timm
from torch.optim.lr_scheduler import CosineAnnealingLR
import optuna

# --- Settings ---
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.backends.cudnn.benchmark = True
BATCH_SIZE = 32
NUM_CLASSES = 3         # Set to your number of classes
NUM_EPOCHS = 4
NUM_WORKERS = 8
PATIENCE = 5            # Early stopping patience




# --- Training Function with Best Model Saving ---
def train_model(model, optimizer, criterion, scheduler, save_path):
    best_acc = 0
    best_state = None
    trigger_times = 0

    for epoch in range(NUM_EPOCHS):
        model.train()
        total_train_loss, correct_train, total_train = 0, 0, 0

        for inputs, targets in train_loader:
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()

            total_train_loss += loss.item()
            _, predicted = outputs.max(1)
            total_train += targets.size(0)
            correct_train += predicted.eq(targets).sum().item()

        scheduler.step()
        train_acc = 100. * correct_train / total_train

        # --- Validation ---
        model.eval()
        correct_val, total_val = 0, 0
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
                outputs = model(inputs)
                _, predicted = outputs.max(1)
                total_val += targets.size(0)
                correct_val += predicted.eq(targets).sum().item()

        val_acc = 100. * correct_val / total_val
        print(f'Epoch [{epoch+1}/{NUM_EPOCHS}] Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%')

        # --- Save Best Model ---
        if val_acc > best_acc:
            best_acc = val_acc
            # For DataParallel use model.module.state_dict(), otherwise model.state_dict()
            best_state = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
            torch.save(best_state, save_path)
            print(f"Best model updated (Val Acc: {best_acc:.2f}%) and saved to {save_path}")
            trigger_times = 0
        else:
            trigger_times += 1
            if trigger_times >= PATIENCE:
                print("Early stopping triggered!")
                break

    return train_acc, best_acc

# --- Optuna Hyperparameter Objective for swinv2_base_window12_192_22k ---
def objective(trial):
    model_name = "swinv2_base_window12_192_22k"
    model = timm.create_model(model_name, pretrained=True, num_classes=NUM_CLASSES)
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
    model.to(DEVICE)

    optimizer_name = trial.suggest_categorical('optimizer', ['AdamW', 'Lion'])
    lr = trial.suggest_float('lr', 1e-5, 1e-3, log=True)
    weight_decay = trial.suggest_float('weight_decay', 1e-6, 1e-2, log=True)

    if optimizer_name == 'AdamW':
        optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    else:
        from lion_pytorch import Lion
        optimizer = Lion(model.parameters(), lr=lr, weight_decay=weight_decay)

    scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
    label_smoothing = trial.suggest_float('label_smoothing', 0.0, 0.2)
    criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)

    # Save the best model from this trial to "best.pt"
    best_model_path = "swinv2_base_window12_192_22k_dummy.pt"
    train_acc, val_acc = train_model(model, optimizer, criterion, scheduler, save_path=best_model_path)
    return val_acc

# --- Run Optuna Hyperparameter Optimization ---
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=10)

print("Best trial:")
best_trial = study.best_trial
print(f"  Value: {best_trial.value}")
print("  Params: ")
for key, value in best_trial.params.items():
    print(f"    {key}: {value}")




[I 2025-04-14 07:32:09,882] A new study created in memory with name: no-name-4d867162-a238-4f81-8c87-3a8c3b104973


Epoch [1/4] Train Acc: 35.56% | Val Acc: 20.44%
Best model updated (Val Acc: 20.44%) and saved to swinv2_base_window12_192_22k_dummy.pt
Epoch [2/4] Train Acc: 57.75% | Val Acc: 81.02%
Best model updated (Val Acc: 81.02%) and saved to swinv2_base_window12_192_22k_dummy.pt
Epoch [3/4] Train Acc: 74.27% | Val Acc: 77.37%


[I 2025-04-14 07:35:37,524] Trial 0 finished with value: 81.02189781021897 and parameters: {'optimizer': 'AdamW', 'lr': 0.00018051043750226707, 'weight_decay': 8.230277592319276e-05, 'label_smoothing': 0.15487782073095058}. Best is trial 0 with value: 81.02189781021897.


Epoch [4/4] Train Acc: 82.05% | Val Acc: 80.29%
Epoch [1/4] Train Acc: 84.23% | Val Acc: 94.16%
Best model updated (Val Acc: 94.16%) and saved to swinv2_base_window12_192_22k_dummy.pt
Epoch [2/4] Train Acc: 95.63% | Val Acc: 98.54%
Best model updated (Val Acc: 98.54%) and saved to swinv2_base_window12_192_22k_dummy.pt
Epoch [3/4] Train Acc: 98.70% | Val Acc: 99.27%
Best model updated (Val Acc: 99.27%) and saved to swinv2_base_window12_192_22k_dummy.pt


[I 2025-04-14 07:39:03,182] Trial 1 finished with value: 99.27007299270073 and parameters: {'optimizer': 'AdamW', 'lr': 1.2001731934016472e-05, 'weight_decay': 0.00016276664836893816, 'label_smoothing': 0.0051024654112364455}. Best is trial 1 with value: 99.27007299270073.


...

Best model updated (Val Acc: 68.61%) and saved to swinv2_base_window12_192_22k_dummy.pt
Epoch [1/4] Train Acc: 34.47% | Val Acc: 68.61%
Best model updated (Val Acc: 68.61%) and saved to swinv2_base_window12_192_22k_dummy.pt
Epoch [2/4] Train Acc: 35.09% | Val Acc: 68.61%
Epoch [3/4] Train Acc: 35.84% | Val Acc: 68.61%


[I 2025-04-14 08:02:30,202] Trial 8 finished with value: 68.61313868613139 and parameters: {'optimizer': 'Lion', 'lr': 0.00016038341869792632, 'weight_decay': 0.0008187704063191017, 'label_smoothing': 0.021522219688769308}. Best is trial 1 with value: 99.27007299270073.


Epoch [4/4] Train Acc: 36.18% | Val Acc: 10.95%
Epoch [1/4] Train Acc: 72.01% | Val Acc: 91.24%
Best model updated (Val Acc: 91.24%) and saved to swinv2_base_window12_192_22k_dummy.pt
Epoch [2/4] Train Acc: 90.99% | Val Acc: 94.89%
Best model updated (Val Acc: 94.89%) and saved to swinv2_base_window12_192_22k_dummy.pt
Epoch [3/4] Train Acc: 96.86% | Val Acc: 88.32%
Epoch [4/4] Train Acc: 98.02% | Val Acc: 99.27%


[I 2025-04-14 08:05:55,079] Trial 9 finished with value: 99.27007299270073 and parameters: {'optimizer': 'AdamW', 'lr': 7.579294898673645e-05, 'weight_decay': 7.268073414499874e-05, 'label_smoothing': 0.16579466161931744}. Best is trial 1 with value: 99.27007299270073.


Best model updated (Val Acc: 99.27%) and saved to swinv2_base_window12_192_22k_dummy.pt
Best trial:
  Value: 99.27007299270073
  Params: 
    optimizer: AdamW
    lr: 1.2001731934016472e-05
    weight_decay: 0.00016276664836893816
    label_smoothing: 0.0051024654112364455
